# 01 · Get the data

Downloads the raw data and saves it in `data/raw/`. Nothing is cleaned here.

| Data | Source | Saved as |
|---|---|---|
| Daily prices (Open, High, Low, Close, Volume) of 6 European defence stocks and the EURO STOXX 50 | Yahoo Finance, via `yfinance` | `data/raw/prices/<name>.csv` |
| Daily Geopolitical Risk index: total, Acts, Threats | Caldara & Iacoviello, matteoiacoviello.com/gpr.htm | `data/raw/gpr_daily.csv` |

**Reproducibility.** Files already in `data/raw/` are *not* downloaded again, so the analysis always uses the same data. Set `REDOWNLOAD = True` only to refresh them on purpose. Prices are downloaded with a fixed end date (`END` in `helpers.py`).

In [1]:
import urllib.request

import pandas as pd
import yfinance as yf

from helpers import TICKERS, START, END, GPR_URL, GPR_XLS, GPR_CSV, price_path

REDOWNLOAD = False

## Prices

`auto_adjust=True` returns prices adjusted for splits and dividends, so a stock split does not look like a crash.

In [2]:
for t in TICKERS:
    path = price_path(t)
    if path.exists() and not REDOWNLOAD:
        print(f"{t:10s} already in {path.name}, skipped")
        continue
    p = yf.download(t, start=START, end=END, auto_adjust=True, progress=False)
    if isinstance(p.columns, pd.MultiIndex):        # newer yfinance returns two header levels
        p.columns = p.columns.get_level_values(0)
    p.index.name = "Date"
    p[["Open", "High", "Low", "Close", "Volume"]].to_csv(path)
    print(f"{t:10s} {p.index.min().date()} -> {p.index.max().date()}  {len(p):5d} rows  -> {path.name}")

RHM.DE     2005-01-03 -> 2026-10-30   5552 rows  -> rheinmetall.csv
LDO.MI     2005-07-18 -> 2026-10-30   5410 rows  -> leonardo.csv
BA.L       2005-01-03 -> 2026-10-30   5524 rows  -> bae.csv
HO.PA      2005-01-03 -> 2026-10-30   5589 rows  -> thales.csv
SAAB-B.ST  2005-01-03 -> 2026-10-30   5503 rows  -> saab.csv
AM.PA      2005-01-03 -> 2026-10-30   5589 rows  -> dassault.csv
^STOXX50E  2007-03-30 -> 2026-10-30   4911 rows  -> stoxx50.csv


## Geopolitical Risk index

The daily file is published as `.xls`. It is converted once to a plain CSV with the three series used in the project:

- `gpr`: total daily index (GPRD)
- `acts`: articles about acts: wars starting, attacks (GPRD_ACT)
- `threats`: articles about threats: tensions, risk of war (GPRD_THREAT)

The index is normalised so that its 1985–2019 average is 100.

In [3]:
if not GPR_XLS.exists() or REDOWNLOAD:
    urllib.request.urlretrieve(GPR_URL, GPR_XLS)
    print("GPR downloaded")

g = pd.read_excel(GPR_XLS)
g["date"] = pd.to_datetime(g["DAY"].astype(str), format="%Y%m%d")
g = g.rename(columns={"GPRD": "gpr", "GPRD_ACT": "acts", "GPRD_THREAT": "threats"})
g = g[["date", "gpr", "acts", "threats"]].dropna()
g.to_csv(GPR_CSV, index=False)
print(f"GPR {g.date.min().date()} -> {g.date.max().date()}  {len(g)} days  -> {GPR_CSV.name}")

GPR 1985-01-01 -> 2026-10-05  15253 days  -> gpr_daily.csv
